# 02 · Extract fields with `ai_extract`

Feed the materialized `parsed_text` into **`ai_extract`** to pull the certificate fields.
`ai_extract` works on **already-parsed text** (it can't read PDF bytes — that's why
`ai_parse_document` runs first). We use **v2.1** with **per-field citations + confidence**
so low-confidence fields can be routed to a human reviewer.

In [ ]:
from config import CATALOG, SCHEMA
import json
FIELDS = [
    "document_type", "insurer_name", "insured_name", "policy_number",
    "effective_date", "expiration_date", "coverage_types",
]
FIELDS_JSON = json.dumps(FIELDS)

## 1 · Extract → materialized `extracted_fields` / `extracted_flat`

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.extracted_fields AS
    SELECT filename,
           ai_extract(full_text, '{FIELDS_JSON}',
               options => map('version', '2.1',
                              'enableCitations', 'true',
                              'enableConfidenceScores', 'true')) AS extracted
    FROM {CATALOG}.{SCHEMA}.parsed_text
""")

value_cols = ',\n'.join([f"extracted:response:{f}:value::string AS {f}" for f in FIELDS])
conf_cols  = ',\n'.join([f"extracted:response:{f}:confidence_score::double AS {f}_conf" for f in FIELDS])
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.extracted_flat AS
    SELECT filename, {value_cols}, {conf_cols}
    FROM {CATALOG}.{SCHEMA}.extracted_fields
""")
print("Extracted:", spark.table(f"{CATALOG}.{SCHEMA}.extracted_flat").count(), "docs")
display(spark.sql(f"""
    SELECT filename, document_type, insurer_name, policy_number,
           effective_date, expiration_date, substr(coverage_types,1,50) AS coverage_types
    FROM {CATALOG}.{SCHEMA}.extracted_flat
"""))

## 2 · Human-in-the-loop review queue

Confidence scores let you auto-accept high-confidence extractions and route the rest to a
reviewer — the straight-through-processing pattern.

In [ ]:
THRESH = 0.80
conf_list = ", ".join([f"coalesce({f}_conf, 0)" for f in FIELDS])
review = spark.sql(f"""
    SELECT filename, round(least({conf_list}), 3) AS min_field_confidence
    FROM {CATALOG}.{SCHEMA}.extracted_flat
    WHERE least({conf_list}) < {THRESH}
    ORDER BY min_field_confidence
""")
print(f"{review.count()} docs have a field below {THRESH} confidence -> review queue")
display(review)

_Next: `03_evaluate_mlflow`._